# Evaluate the performance of different GNNs across different adjacencies and input matrices. Table 3 from the paper

In [ ]:
# --- Core data handling ---
import json
import numpy as np
import pandas as pd
import geopandas as gpd

# --- Plotting ---
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch, Patch
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

# --- Basemaps ---
import contextily as ctx

# --- Statistics ---
from scipy.stats import (
    ttest_rel,
    wilcoxon,
    shapiro,
    rankdata,
    pearsonr,
    spearmanr,
    linregress,
    gaussian_kde,
)

# --- Visualization helpers ---
import seaborn as sns

# --- Spatial statistics ---
from libpysal.weights import W
from esda.moran import Moran_Local


# Functions

In [ ]:
def plot_county_lisa_map(
    transmit_matrix,
    mse_final: gpd.GeoDataFrame,
    counties: gpd.GeoDataFrame,
    column: str = "mean_mse",
    permutations: int = 999,
    sig: float = 0.05,
    figsize=(12, 8),
    title="Local Moran’s I cluster map",
    add_basemap: bool = True,
    add_scale_and_north: bool = True,
    plot_arrows: bool = False,
    arrow_color: str = "black",
    arrow_scale: float = 10.0,
):
    """
    Plot a LISA (Local Moran's I) cluster map for 'column'.
    Only statistically significant clusters are colored. Non significant areas remain gray.

    Parameters
    ----------
    transmit_matrix : pandas.DataFrame or numpy.ndarray
        Row i, col j represents spatial weight from i to j. Used to build W.
        If a pandas DataFrame, its index and columns should match the order of 'mse_final'.
    mse_final : GeoDataFrame
        GeoDataFrame with 'column' and geometry for the analysis unit.
    counties : GeoDataFrame
        Background geometries to provide map context. Will be reprojected if needed.
    column : str
        Column in mse_final on which to compute LISA.
    permutations : int
        Number of permutations for significance testing.
    sig : float
        Significance threshold (e.g., 0.05).
    figsize : tuple
        Figure size.
    title : str
        Plot title.
    add_basemap : bool
        Add CartoDB Positron basemap.
    add_scale_and_north : bool
        Add scale bar and north arrow.
    plot_arrows : bool
        If True, draws arrows following the transmit_matrix strengths as alpha.
    arrow_color : str
        Arrow color.
    arrow_scale : float
        Arrow size scaling factor.
    """

    # Work on copies and ensure Web Mercator for consistent display with contextily
    g = mse_final.copy()
    bg = counties.copy()

    if g.crs is None or g.crs.is_geographic:
        g = g.to_crs(epsg=3857)
    if bg.crs is None or bg.crs.is_geographic:
        bg = bg.to_crs(epsg=3857)

    # Build spatial weights W from the provided transmit_matrix
    # If transmit_matrix is a DataFrame, align to mse_final order when possible
    if hasattr(transmit_matrix, "values"):
        # Assume DataFrame with matching index/columns or at least same ordering length
        A = transmit_matrix.values
    else:
        A = np.asarray(transmit_matrix)

    # Defensive checks
    n = len(g)
    if A.shape[0] != n or A.shape[1] != n:
        raise ValueError(
            f"Adjacency shape {A.shape} does not match number of rows in mse_final {n}."
        )

    # Convert adjacency to PySAL W, then row-standardize
    # libpysal expects a dict-of-lists neighbor structure. Build from nonzero entries.
    neighbors = {i: list(np.where(A[i] != 0)[0]) for i in range(n)}
    weights = {
        i: list(A[i, neighbors[i]].astype(float)) if len(neighbors[i]) > 0 else []
        for i in range(n)
    }
    w = W(neighbors=neighbors, weights=weights, silence_warnings=True)
    w.transform = "R"  # row-standardized

    # Variable, standardized
    y = g[column].to_numpy().astype(float)
    y_mean = y.mean()
    y_std = y.std(ddof=1) if y.std(ddof=1) != 0 else 1.0
    z = (y - y_mean) / y_std

    # Local Moran's I
    lisa = Moran_Local(z, w, permutations=permutations)

    # Significance mask
    sig_mask = lisa.p_sim < sig

    # Quadrants in PySAL:
    # 1 = HH, 2 = LH, 3 = LL, 4 = HL (based on Moran scatterplot quadrants)
    quad = lisa.q

    # Assign cluster labels and colors only where significant
    labels = np.array(["Not significant"] * n, dtype=object)
    colors = np.array(["#d9d9d9"] * n, dtype=object)  # light gray for non significant

    # Typical LISA cluster colors
    # HH: red, LL: blue, LH: light blue, HL: pink
    color_map = {
        1: ("High-High", "#e31a1c"),   # red
        2: ("Low-High",  "#a6cee3"),   # light blue
        3: ("Low-Low",   "#1f78b4"),   # blue
        4: ("High-Low",  "#fb9a99"),   # pink
    }

    for i in range(n):
        if sig_mask[i]:
            lbl, col = color_map[quad[i]]
            labels[i] = lbl
            colors[i] = col

    g["_lisa_label"] = labels
    g["_lisa_color"] = colors

    # Set up figure
    fig, ax = plt.subplots(figsize=figsize)

    # Background counties in light gray
    bg.plot(ax=ax, facecolor="#f0f0f0", edgecolor="white", linewidth=0.2)

    # Plot significant clusters by color, leave non significant in gray outline
    # We will draw non significant as thin outlines only, then draw significant filled on top
    g[~sig_mask].plot(ax=ax, facecolor="#f7f7f7", edgecolor="#bbbbbb", linewidth=0.3)

    for quad_val, (lbl, col) in color_map.items():
        sel = (g["_lisa_label"] == lbl)
        if sel.any():
            g[sel].plot(ax=ax, facecolor=col, edgecolor="black", linewidth=0.4, label=lbl)

    # Optional transmit arrows (kept from your original function)
    if plot_arrows:
        centroids = g.geometry.centroid
        # Normalize alphas from nonzero strengths
        non_zero_vals = A[A > 0]
        vmin_arrow, vmax_arrow = (non_zero_vals.min(), non_zero_vals.max()) if non_zero_vals.size > 0 else (0, 1)

        for i in range(n):
            for j in range(n):
                strength = A[i, j]
                if strength > 0:
                    start = centroids.iloc[i]
                    end = centroids.iloc[j]
                    alpha = (strength - vmin_arrow) / (vmax_arrow - vmin_arrow) if vmax_arrow > vmin_arrow else 0.5
                    alpha = 0.1 + alpha * 0.9
                    arrow = FancyArrowPatch(
                        (start.x, start.y), (end.x, end.y),
                        color=arrow_color,
                        arrowstyle='->',
                        mutation_scale=arrow_scale,
                        linewidth=1,
                        alpha=alpha
                    )
                    ax.add_patch(arrow)

    # Basemap
    if add_basemap:
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

    # Legend without colorbar
    handles = []
    for quad_val, (lbl, col) in color_map.items():
        count = int((g["_lisa_label"] == lbl).sum())
        if count > 0:
            handles.append(Patch(facecolor=col, edgecolor="black", label=f"{lbl} (n={count})"))
    # Add a gray swatch only if you want to show non significant in legend
    # non_sig_count = int((~sig_mask).sum())
    # handles.append(Patch(facecolor="#f7f7f7", edgecolor="#bbbbbb", label=f"Not significant (n={non_sig_count})"))

    if handles:
        leg = ax.legend(
            handles=handles,
            title=f"LISA clusters (p < {sig})",
            loc="lower right",
            frameon=True,
            facecolor="white",
            edgecolor="#cccccc",
            framealpha=0.4
        )
        for text in leg.get_texts():
            text.set_fontsize(10)
          # reduces vertical spacing between legend entries

    # Scale bar and north arrow
    if add_scale_and_north:
        scalebar = AnchoredSizeBar(
            ax.transData,
            500_000,  # 500 km in Web Mercator units
            '500 km',
            'lower left',
            pad=1,
            color='black',
            frameon=False,
            size_vertical=50_000,
            fontproperties=fm.FontProperties(size=12),
            fill_bar=True
        )
        ax.add_artist(scalebar)

        # Position a simple north arrow near the scale bar
        fig.canvas.draw()
        bbox = scalebar.get_window_extent(fig.canvas.renderer)
        mid_x = bbox.x0 + 0.25 * (bbox.x1 - bbox.x0)
        mid_y = (bbox.y0 + bbox.y1) / 2
        inv = ax.transData.inverted()
        x_pos, y_pos = inv.transform((mid_x, mid_y))
        y_pos += 150000
        x_pos += 150000

        ax.annotate('N', xy=(x_pos, y_pos), ha='center', va='center', fontsize=22, color='black')
        arrow_start_y = y_pos + 220000
        arrow_length = 120000
        ax.annotate(
            '',
            xy=(x_pos, arrow_start_y + arrow_length),
            xytext=(x_pos, arrow_start_y),
            arrowprops=dict(facecolor='black', width=4, headwidth=14, headlength=18)
        )

    ax.set_title(title, fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

def plot_county_influence_map(
    transmit_matrix,
    mse_final,
    counties,
    column='mean_mse',
    plot_arrows=True,
    arrow_color='red',
    cmap='RdBu_r',  # diverging cmap centered at zero
    figsize=(12, 8),
    title="County Influence Arrows (Alpha = Scaled Strength)"
):
    # Prepare adjacency matrix
    adj = transmit_matrix.fillna(0).values if hasattr(transmit_matrix, "fillna") else np.array(transmit_matrix)

    # Ensure projected CRS
    best_counties = mse_final.copy()
    if best_counties.crs.is_geographic:
        best_counties = best_counties.to_crs(epsg=3857)

    if counties.crs.is_geographic:
        counties = counties.to_crs(epsg=3857)

    # Set up figure
    fig, ax = plt.subplots(figsize=figsize)

    # Plot background counties
    counties.plot(ax=ax, facecolor='lightgray', edgecolor='white', linewidth=0.2)

    # Determine symmetric vmin and vmax around zero for coloring
    max_abs = np.max(np.abs(best_counties[column]))
    vmin, vmax = -max_abs, max_abs

    # Plot colored MSE counties with diverging cmap centered on zero
    best_counties.plot(
        ax=ax,
        column=column,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        legend=True,
        edgecolor='black',
        linewidth=0.3,
        legend_kwds={'shrink': 0.6, 'label': "Mean MSE"}
    )

    # Get centroids
    centroids = best_counties.geometry.centroid

    if plot_arrows:
        # Normalize alpha values
        non_zero_vals = adj[adj > 0]
        vmin_arrow, vmax_arrow = (non_zero_vals.min(), non_zero_vals.max()) if non_zero_vals.size > 0 else (0, 1)

        for i in range(len(centroids)):
            for j in range(len(centroids)):
                strength = adj[i, j]
                if strength > 0:
                    start = centroids.iloc[i]
                    end = centroids.iloc[j]
                    alpha = (strength - vmin_arrow) / (vmax_arrow - vmin_arrow) if vmax_arrow > vmin_arrow else 0.5
                    alpha = 0.1 + alpha * 0.9  # Stretch to [0.1, 1.0]
                    arrow = FancyArrowPatch(
                        (start.x, start.y), (end.x, end.y),
                        color=arrow_color,
                        arrowstyle='->',
                        mutation_scale=10,
                        linewidth=1,
                        alpha=alpha
                    )
                    ax.add_patch(arrow)

    # Add basemap
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
    
    
    # Add scale bar
    scalebar = AnchoredSizeBar(
        ax.transData,
        500_000,  # 500 km in map units
        '500 km',
        'lower left',
        pad=1,
        color='black',
        frameon=False,
        size_vertical=50_000,
        fontproperties=fm.FontProperties(size=12),
        fill_bar=True
    )
    ax.add_artist(scalebar)

    # Draw and get bounding box of scalebar
    fig.canvas.draw()  # force rendering so bbox exists
    bbox = scalebar.get_window_extent(fig.canvas.renderer)

    # Instead of full midpoint (text + bar), bias toward left = center of bar
    mid_x = bbox.x0 + 0.25 * (bbox.x1 - bbox.x0)  # adjust factor here
    mid_y = (bbox.y0 + bbox.y1) / 2

    # Convert back to data coordinates
    inv = ax.transData.inverted()
    x_pos, y_pos = inv.transform((mid_x, mid_y))

    # Shift north arrow above the scalebar midpoint
    y_pos += 150000  # vertical offset
    x_pos += 150000   # adjust this number until it looks perfect

    # Place "N"
    ax.annotate(
        'N',
        xy=(x_pos, y_pos),
        ha='center', va='center',
        fontsize=28, color='black'
    )

    # Draw arrow above "N"
    arrow_start_y = y_pos + 250000
    arrow_length = 120000
    ax.annotate(
        '',
        xy=(x_pos, arrow_start_y + arrow_length),
        xytext=(x_pos, arrow_start_y),
        arrowprops=dict(facecolor='black', width=5, headwidth=15, headlength=20)
    )

    # Clean layout
    ax.set_title(title, fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

def print_model_comparison(errors_a, errors_b, name_a, name_b, alpha=0.05, print_latex=True):
    results = compare_model_errors(errors_a, errors_b, alpha=alpha)

    print("\n📊 Model Comparison Report")
    print("─────────────────────────────")

    print(f"\n🔍 Comparing {name_a} vs {name_b}")

    # 1. Normality Check
    print(f"\n📌 Normality of error differences:")
    print(f" → Shapiro-Wilk p-value: {results['normality_p']:.4e}")
    print(f" → Are differences normally distributed? {'✅ Yes' if results['normality'] == 'normal' else '❌ No'}")

    # 2. Main Test Result
    print(f"\n🔬 Statistical Test Used:")
    print(f" → {results['test_used']}")
    print(f" → Null Hypothesis: {name_a} errors are NOT significantly greater than {name_b} errors.")
    print(f" → Test statistic: {results['test_statistic']:.3f}")
    print(f" → p-value: {results['p_value']:.4e}")

    if results['p_value'] < alpha:
        print(f" ✅ Result: {name_a} errors are significantly **greater** than {name_b} errors.")
    else:
        print(f" ❌ No significant difference → checking reverse...")
        if 'reverse_p_value' in results:
            print(f" → Reverse test p-value: {results['reverse_p_value']:.4e}")
            if results['reverse_p_value'] < alpha:
                print(f" ✅ Reverse result: {name_b} errors are significantly **greater** than {name_a} errors.")
            else:
                print(f" ❌ Still no significant difference.")

    # 3. Effect Size
    effect = results['effect_size']
    label = results['effect_type']
    print(f"\n📐 Effect Size: {label} = {effect:.3f}")

    # 4. Mean / Median Summary
    print(f"\n📈 Mean difference ({name_a} − {name_b}): {results['mean_diff']:.4f}")
    print(f"📉 Median difference ({name_a} − {name_b}): {results['median_diff']:.4f}")
    print("─────────────────────────────\n")

    # ---------------- LaTeX report-style table ----------------
    latex_table = r"""
\begin{table}[ht]
\centering
\caption{Comparison of model errors between """ + name_a + " and " + name_b + r"""}
\begin{tabular}{l l}
\hline
\textbf{Section} & \textbf{Result} \\
\hline
\multicolumn{2}{l}{\textbf{Normality Test}} \\
Shapiro--Wilk p-value & """ + f"{results['normality_p']:.4e}" + r""" \\
Normality assumption & """ + ("Yes (data approx. normal)" if results['normality']=="normal" else "No (non-normal distribution)") + r""" \\
\hline
\multicolumn{2}{l}{\textbf{Main Statistical Test}} \\
Test used & """ + results['test_used'] + r""" \\
Null hypothesis & """ + name_a + " errors are not greater than " + name_b + " errors" + r""" \\
Test statistic & """ + f"{results['test_statistic']:.3f}" + r""" \\
p-value & """ + f"{results['p_value']:.4e}" + r""" \\
Result & """ + (f"{name_a} errors are significantly greater than {name_b} errors"
                 if results['p_value'] < alpha
                 else f"No significant difference between {name_a} and {name_b}") + r""" \\
\hline
\multicolumn{2}{l}{\textbf{Effect Size}} \\
Measure & """ + f"{results['effect_type']} = {results['effect_size']:.3f}" + r""" \\
\hline
\multicolumn{2}{l}{\textbf{Summary of Differences}} \\
Mean difference ($""" + name_a + r""" - """ + name_b + r"""$) & """ + f"{results['mean_diff']:.4f}" + r""" \\
Median difference ($""" + name_a + r""" - """ + name_b + r"""$) & """ + f"{results['median_diff']:.4f}" + r""" \\
"""

    if 'reverse_p_value' in results:
        latex_table += r"""\hline
\multicolumn{2}{l}{\textbf{Reverse Hypothesis Test}} \\
Reverse p-value & """ + f"{results['reverse_p_value']:.4e}" + r""" \\
Reverse effect size & """ + f"{results['reverse_effect_type']} = {results['reverse_effect_size']:.3f}" + r""" \\
"""

    latex_table += r"""\hline
\end{tabular}
\end{table}
"""
    if print_latex:
        print("\nLaTeX Report-Style Table:\n")
        print(latex_table)

def plot_mse_distributions(mse_a, mse_b, model_a="Model A", model_b="Model B"):
    # Extract mean MSE values
    mse_vals_a = mse_a["mean_mse"].dropna().values
    mse_vals_b = mse_b["mean_mse"].dropna().values

    # Create KDEs
    kde_a = gaussian_kde(mse_vals_a)
    kde_b = gaussian_kde(mse_vals_b)

    # Define x-axis range
    all_vals = np.concatenate([mse_vals_a, mse_vals_b])
    x_vals = np.linspace(all_vals.min(), all_vals.max(), 500)

    # Plot
    plt.figure(figsize=(10, 6))

    # Histograms
    plt.hist(mse_vals_a, bins=30, color='blue', alpha=0.4, density=True, label=f'{model_a} - Histogram')
    plt.hist(mse_vals_b, bins=30, color='red', alpha=0.4, density=True, label=f'{model_b} - Histogram')

    # KDEs
    plt.plot(x_vals, kde_a(x_vals), color='blue', label=f'{model_a} - KDE')
    plt.plot(x_vals, kde_b(x_vals), color='red', label=f'{model_b} - KDE')

    # Fill under KDE curves
    plt.fill_between(x_vals, 0, kde_a(x_vals), color='blue', alpha=0.2)
    plt.fill_between(x_vals, 0, kde_b(x_vals), color='red', alpha=0.2)

    # Labels and layout
    plt.title(f"{model_a} vs {model_b} MSE Distribution")
    plt.xlabel("MSE Value")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def build_comparison_summary_latex(
    rows,
    caption="Summary of Model Comparison Results with Population Correlation and Interpretation",
    label="tab:model_comparison_summary",
    colspec=r"p{0.6cm} p{1.7cm} p{1cm} p{1.4cm} p{6cm}",
    interp_width="6cm",
    add_note=True,
):
    """
    Build a LaTeX table similar to the extended hypothesis table.

    Parameters
    ----------
    rows : list of dict
        Each dict is expected to have at least:
          - "ID": group identifier (e.g. "H1")
          - "mse_comparison_latex": LaTeX string for MSE comparison (e.g. "M1 > M4")
          - "effect_rrb": float, effect size
          - "pop_beta": float, population slope (may be np.nan)
          - "pop_beta_stars": string with significance stars (e.g. "***" or "")
          - "interpretation": string with interpretation text for the group
            (same for all rows of a given ID, only used on the first row via \\multirow)
    caption, label : str
        LaTeX caption and label.
    colspec : str
        LaTeX tabular column specification.
    interp_width : str
        Width for the multirow interpretation column (e.g. "6cm").
    add_note : bool
        If True, adds the note and significance codes block below the table.

    Returns
    -------
    str
        LaTeX code for the table.
    """

    if not rows:
        return ""

    lines = []
    lines.append(r"\begin{table}[ht]")
    lines.append(r"\centering")
    lines.append(r"\caption{" + caption + r"}")
    lines.append(r"\label{" + label + r"}")
    lines.append(r"\begin{tabular}{" + colspec + r"}")
    lines.append(r"\hline")
    lines.append(
        r"\textbf{ID} & \textbf{MSE Comparison} & \textbf{Effect Size $r_{rb}$} "
        r"& \textbf{Pop. Slope $\beta$} & \textbf{Interpretation} \\"
    )
    lines.append(r"\hline")

    # Group rows by consecutive IDs
    groups = []
    current_group = [0]
    last_id = rows[0].get("ID", "")
    for i in range(1, len(rows)):
        id_i = rows[i].get("ID", "")
        if id_i == last_id:
            current_group.append(i)
        else:
            groups.append((last_id, current_group))
            current_group = [i]
            last_id = id_i
    groups.append((last_id, current_group))

    # Emit rows with multirow for ID and Interpretation
    for group_id, idx_list in groups:
        n = len(idx_list)

        for j, idx in enumerate(idx_list):
            row = rows[idx]

            # First column: ID or multirow
            if n > 1 and j == 0:
                id_cell = rf"\multirow[t]{{{n}}}{{*}}{{{group_id}}}"
            elif n > 1:
                id_cell = ""  # continuation row
            else:
                id_cell = group_id

            comp = row["mse_comparison_latex"]
            eff = f"{row['effect_rrb']:.3f}"

            if np.isnan(row["pop_beta"]):
                beta_str = ""
            else:
                beta_str = f"{row['pop_beta']:.4f}{row.get('pop_beta_stars', '')}"

            # Interpretation column: only on first row of group if grouped
            interp_text = row.get("interpretation", "")
            if n > 1 and j == 0:
                interp_cell = rf"\multirow[t]{{{n}}}{{{interp_width}}}{{{interp_text}}}"
            elif n > 1:
                interp_cell = ""
            else:
                interp_cell = interp_text

            lines.append(f"{id_cell} & {comp} & {eff} & {beta_str} & {interp_cell} \\\\")

        # Optional single \hline between groups, you can comment this out if you do not want it
        lines.append(r"\hline")

    # Replace last single \hline with \hline\hline to match your example
    if len(lines) >= 1 and lines[-1] == r"\hline":
        lines[-1] = r"\hline\hline"

    lines.append(r"\end{tabular}")

    if add_note:
        lines.append(r"\\[3pt]")
        lines.append(r"\raggedright")
        lines.append(r"\footnotesize")
        lines.append(
            r"Note: $\beta$ denotes the regression slope between county population (log scale) "
            r"and MSE difference."
        )
        lines.append(
            r"Significance codes: * $p<0.05$, ** $p<0.01$, *** $p<0.001$, "
            r"$\approx$ = no significant difference."
        )

    lines.append(r"\end{table}")

    return "\n".join(lines)

def p_to_stars(p, levels=(0.05, 0.01, 0.001)):
    """
    Map a p value to significance stars.
    """
    if p is None or np.isnan(p):
        return ""
    if p < levels[2]:
        return "***"
    if p < levels[1]:
        return "**"
    if p < levels[0]:
        return "*"
    return ""

# ---- helper to preserve acronyms like MSE/RMSE/MAE/R2/AUC ----
def prettify_label(name: str) -> str:
    acronyms = {"MSE", "RMSE", "MAE", "MAPE", "R2", "R^2", "AUC"}
    words = name.replace("_", " ").split()
    pretty = []
    for w in words:
        if w.upper() in acronyms:
            pretty.append(w.upper())
        else:
            pretty.append(w.capitalize())
    return " ".join(pretty)

def plot_centrality_vs_mse_seaborn_stats(
    df,
    centrality_col: str,
    mse_col: str = "mean_mse_diff",
    kind: str = "scatter",
    log_transform: str = "none",
    figsize=(7, 7),
    cmap="Blues",
    bins=30,
    alpha=0.6,
    reg_color="crimson",
    reg_ci=95,
    stats_engine: str = "linregress",
    robust_se: str = None,
    dpi: int = 300,
    save_path: str = None,
    plot_figure: bool = True,
):
    """
    Same as plot_centrality_vs_mse_seaborn, but returns a dict with
    beta, p_beta, Pearson and Spearman stats.
    """
    df_plot = df.copy()

    # Safe log transforms
    if log_transform in ["x", "both"]:
        df_plot[centrality_col] = np.log1p(df_plot[centrality_col])
    if log_transform in ["y", "both"]:
        df_plot[mse_col] = np.sign(df_plot[mse_col]) * np.log1p(np.abs(df_plot[mse_col]))

    # Drop rows with NaN or inf after transforms
    df_plot = df_plot.replace([np.inf, -np.inf], np.nan).dropna(subset=[centrality_col, mse_col])

    # Correlations on original scale (no transforms)
    pearson_r, pearson_p = pearsonr(df[centrality_col], df[mse_col])
    spearman_rho, spearman_p = spearmanr(df[centrality_col], df[mse_col])

    sns.set(style="white", context="talk")
    plt.rcParams["figure.dpi"] = dpi
    plt.rcParams["savefig.dpi"] = dpi

    g = sns.jointplot(
        data=df_plot,
        x=centrality_col,
        y=mse_col,
        kind=kind,
        height=figsize[0],
        ratio=5,
        space=0.2,
        marginal_kws=dict(bins=bins, fill=True, color="steelblue", alpha=0.5),
        color="steelblue" if kind == "scatter" else None,
        cmap=cmap if kind == "hex" else None,
        alpha=alpha
    )

    # Regression line with CI
    sns.regplot(
        data=df_plot,
        x=centrality_col,
        y=mse_col,
        scatter=False,
        ax=g.ax_joint,
        color=reg_color,
        ci=reg_ci,
        line_kws={"lw": 2},
    )

    # Reference line for delta MSE = 0
    g.ax_joint.axhline(0, color="black", linestyle="--", lw=1)

    # Regression slope and p value
    xvals = df_plot[centrality_col].to_numpy()
    yvals = df_plot[mse_col].to_numpy()

    if stats_engine == "linregress":
        lr = linregress(xvals, yvals)
        beta, p_beta = lr.slope, lr.pvalue
    else:
        try:
            import statsmodels.api as sm
            X = sm.add_constant(xvals)
            model = sm.OLS(yvals, X).fit(cov_type=(robust_se if robust_se else "nonrobust"))
            beta, p_beta = float(model.params[1]), float(model.pvalues[1])
        except Exception:
            lr = linregress(xvals, yvals)
            beta, p_beta = lr.slope, lr.pvalue

    # Correlation annotation
    corr_text = (
        f"Pearson r = {pearson_r:.2f} (p = {pearson_p:.3f})\n"
        f"Spearman ρ = {spearman_rho:.2f} (p = {spearman_p:.3f})"
    )
    g.ax_joint.text(
        0.98, 0.98, corr_text,
        transform=g.ax_joint.transAxes,
        fontsize=14,
        va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8)
    )

    # Beta annotation
    beta_text = f"β (slope) = {beta:.3g}\np = {p_beta:.3g}"
    g.ax_joint.text(
        0.98, 0.02, beta_text,
        transform=g.ax_joint.transAxes,
        fontsize=14,
        va="bottom", ha="right",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8)
    )

    pretty_x = prettify_label(centrality_col)
    pretty_y = prettify_label(mse_col)
    g.set_axis_labels(
        pretty_x + (" (log scale)" if log_transform in ["x", "both"] else ""),
        pretty_y + (" (log scale)" if log_transform in ["y", "both"] else ""),
        fontsize=14
    )

    g.fig.suptitle(f"{pretty_y} vs. {pretty_x}", fontsize=16, y=0.98)
    g.fig.tight_layout(pad=1.5)

    if save_path:
        g.fig.savefig(save_path, dpi=dpi, bbox_inches="tight")

    if plot_figure:
        plt.show()

    stats = dict(
        beta=beta,
        p_beta=p_beta,
        pearson_r=pearson_r,
        pearson_p=pearson_p,
        spearman_rho=spearman_rho,
        spearman_p=spearman_p,
    )
    return stats

# ---------- Effect sizes ----------
def cliffs_delta_unpaired(x, y):
    """Standard (unpaired) Cliff's delta: all-vs-all comparisons."""
    x = np.asarray(x)
    y = np.asarray(y)
    nx, ny = len(x), len(y)
    total = 0
    for xi in x:
        for yi in y:
            if xi > yi:
                total += 1
            elif xi < yi:
                total -= 1
    return total / (nx * ny)

# this one is used!
def cureton_rank_biserial_paired(a, b):
    """
    Cureton matched-pairs rank-biserial correlation:
        r = (SF - SU) / S
    where S = N(N+1)/2, ranks are assigned to |a - b| with average ranks for ties,
    SF is the sum of ranks for positive differences, SU for negative.

    Returns
    -------
    r : float          # signed effect size in [-1, 1]
    details : dict     # components for auditing
    """
    a = np.asarray(a)
    b = np.asarray(b)
    d = a - b

    # remove zero differences, as in Wilcoxon
    mask = d != 0
    d = d[mask]
    if d.size == 0:
        return 0.0, {"N": 0, "SF": 0.0, "SU": 0.0, "S": 0.0}

    N = d.size
    # ranks of absolute differences, average ranks for ties
    ranks = rankdata(np.abs(d), method="average")

    SF = ranks[d > 0].sum()  # favorable rank sum
    SU = ranks[d < 0].sum()  # unfavorable rank sum
    S = N * (N + 1) / 2.0    # total possible rank sum

    r = (SF - SU) / S
    return r, {"N": N, "SF": SF, "SU": SU, "S": S}

# ---------- Helper functions ----------
def cliffs_delta(x, y):
    nx, ny = len(x), len(y)
    total = 0
    for xi in x:
        for yi in y:
            if xi > yi:
                total += 1
            elif xi < yi:
                total -= 1
    return total / (nx * ny)

# ---------- Main comparison ----------
def compare_model_errors(errors_a, errors_b, alpha=0.05):
    errors_a = np.asarray(errors_a)
    errors_b = np.asarray(errors_b)
    diffs = errors_a - errors_b
    results = {}

    # Always compute paired rank-biserial and paired Cliff's delta (identical)
    r_rb, _ = cureton_rank_biserial_paired(errors_a, errors_b) #paired_rank_biserial(errors_a, errors_b)
    delta_unpaired = cliffs_delta_unpaired(errors_a, errors_b) # (errors_a, errors_b)

    # Normality test on paired differences
    _, p_norm = shapiro(diffs)
    normal = (p_norm > alpha)
    results['normality_p'] = p_norm
    results['normality'] = "normal" if normal else "non-normal"

    if normal:
        stat, p_val = ttest_rel(errors_a, errors_b, alternative='greater')
        test_used = 'Paired t-test (A > B)'
        mean_diff = np.mean(diffs)
        std_diff = np.std(diffs, ddof=1)
        cohens_d = mean_diff / std_diff if std_diff != 0 else np.nan
        primary_effect = cohens_d
        primary_effect_label = "Cohen's d"
    else:
        # Wilcoxon on diffs is fine
        stat, p_val = wilcoxon(diffs, alternative='greater')
        test_used = 'Wilcoxon signed-rank test (A > B)'
        # Keep your original primary effect for non-normal, but use the PAIRED version
        primary_effect = delta_unpaired
        primary_effect_label = "Paired Cliff's Delta"

    results.update({
        "test_used": test_used,
        "test_statistic": stat,
        "p_value": p_val,
        "mean_diff": np.mean(diffs),
        "median_diff": np.median(diffs),
        # primary effect as before
        "effect_size": primary_effect,
        "effect_type": primary_effect_label,
        # always include both names explicitly
        "rank_biserial": r_rb,
        "unpaired_cliffs_delta": delta_unpaired
    })

    # Reverse check if not significant
    if p_val >= alpha:
        diffs_rev = errors_b - errors_a
        if normal:
            _, p_rev = ttest_rel(errors_b, errors_a, alternative='greater')
            mean_diff_rev = np.mean(diffs_rev)
            std_diff_rev = np.std(diffs_rev, ddof=1)
            cohens_d_rev = mean_diff_rev / std_diff_rev if std_diff_rev != 0 else np.nan
            effect_size_rev = cohens_d_rev
            effect_label_rev = "Cohen's d"
        else:
            _, p_rev = wilcoxon(diffs_rev, alternative='greater')
            # flipping order flips the sign
            effect_size_rev = -delta_unpaired
            effect_label_rev = "Paired Cliff's Delta"

        results.update({
            "reverse_p_value": p_rev,
            "reverse_effect_size": effect_size_rev,
            "reverse_effect_type": effect_label_rev,
            # also provide reverse r_rb for completeness
            "reverse_rank_biserial": -r_rb,
            "reverse_unpaired_cliffs_delta": -delta_unpaired
        })

    return results

def get_model_comparison_summary(
    errors_a,
    errors_b,
    name_a: str,
    name_b: str,
    alpha: float = 0.05,
    print_details: bool = True,
    print_latex: bool = False
):
    """
    Wrapper around compare_model_errors that:
      * optionally prints the detailed report via print_model_comparison
      * returns comparison direction, effect size r_rb, and p values.
    """
    # Use your existing routine to print the nice report if desired
    if print_details or print_latex:
        print_model_comparison(
            errors_a, errors_b,
            name_a, name_b,
            alpha=alpha,
            print_latex=print_latex
        )

    results = compare_model_errors(errors_a, errors_b, alpha=alpha)

    # Decide direction symbol and human readable comparison
    if results["p_value"] < alpha:
        # main test: a errors greater than b errors
        direction_symbol = ">"
        comparison_plain = f"{name_a} > {name_b}"
        comparison_latex = f"{name_a} > {name_b}"
    elif "reverse_p_value" in results and results["reverse_p_value"] < alpha:
        direction_symbol = "<"
        comparison_plain = f"{name_a} < {name_b}"
        comparison_latex = f"{name_a} < {name_b}"
    else:
        direction_symbol = "≈"
        comparison_plain = f"{name_a} ≈ {name_b}"
        comparison_latex = f"{name_a} \\approx {name_b}"

    summary = dict(
        name_a=name_a,
        name_b=name_b,
        comparison_plain=comparison_plain,
        comparison_latex=comparison_latex,
        direction_symbol=direction_symbol,
        rank_biserial=results["rank_biserial"],
        p_value=results["p_value"],
        reverse_p_value=results.get("reverse_p_value", np.nan),
        test_used=results["test_used"],
    )
    return summary

def analyze2_collect_stats(
    model_a,
    model_b,
    hypothesis_id: str = None,
    base_path=r"/../../3_Data/regional_mse (5)/region_mse_with_errors",
    model_specifics="_adamw_0.0001_mse_butterworth.shp",
    transmit_csv="transmit.csv",
    fips_txt="fips.txt",
    counties_shp="counties.shp",
    plot_maps=True,
    print_latex=True,
    print_dist_comp=True,
    plot_distributions=True,
    plot_histogram=True,
    plot_scatter=True,
    plot_populations=True,
    plot_lisa = False,
    alpha: float = 0.05,
):
    """
    High level wrapper like analyze2, but additionally returns a row dict
    with everything needed for the summary table:
      * hypothesis ID
      * model comparison string
      * effect size r_rb
      * population slope beta and its p value
    """
    # Paths
    f_a = f"{base_path}{model_a}{model_specifics}"
    f_b = f"{base_path}{model_b}{model_specifics}"

    # Load data
    mse_a = gpd.read_file(f_a)
    mse_b = gpd.read_file(f_b)
    mse_final = mse_a.copy()

    transmit_matrix = pd.read_csv(transmit_csv, index_col=0)
    transmit_matrix.index = transmit_matrix.index.astype(int)
    transmit_matrix.columns = transmit_matrix.columns.astype(int)
    transmit_matrix = transmit_matrix.loc[mse_final.countyfips, mse_final.countyfips]

    with open(fips_txt, "r") as file:
        mainland_countyfips = json.load(file)

    counties = gpd.read_file(counties_shp)
    counties = counties[counties["countyfips"].astype(str).isin(mainland_countyfips)]

    # Compute diff
    diff_column = f"mse_diff_{model_a} - {model_b}"
    mse_final[diff_column] = mse_a["mean_mse"] - mse_b["mean_mse"]

    # Histogram of differences
    if plot_histogram:
        plt.figure(figsize=(8, 4))
        plt.hist(mse_final[diff_column], bins=30, color="blue", edgecolor="black")
        plt.axvline(0, color="black", linestyle="-", linewidth=2)
        plt.title(f"Histogram of MSE Differences: {model_a} − {model_b}")
        plt.xlabel("MSE difference")
        plt.ylabel("Frequency")
        plt.show()



    # Scatter of raw MSE
    if plot_scatter:
        plt.figure(figsize=(6, 6))
        plt.hexbin(
            mse_a["mean_mse"],
            mse_b["mean_mse"],
            gridsize=40,
            cmap="Purples",
            bins="log"
        )
        max_val = max(mse_a["mean_mse"].max(), mse_b["mean_mse"].max())
        plt.plot([0, max_val], [0, max_val], "k--", lw=1)
        plt.colorbar(label="log(count)")
        plt.title(f"MSE density plot: {model_a} vs {model_b}")
        plt.xlabel(f"{model_a} MSE")
        plt.ylabel(f"{model_b} MSE")
        plt.xlim(0, max_val)
        plt.ylim(0, max_val)
        plt.grid(True)
        plt.show()

    # MSE distributions
    if plot_distributions:
        plot_mse_distributions(mse_a, mse_b, model_a=model_a, model_b=model_b)

    # Detailed model comparison and summary stats
    comparison_summary = get_model_comparison_summary(
        mse_a["mean_mse"],
        mse_b["mean_mse"],
        model_a,
        model_b,
        alpha=alpha,
        print_details=print_dist_comp,
        print_latex=print_latex,
    )

    # Maps
    if plot_maps:
        plot_county_influence_map(
            transmit_matrix=transmit_matrix,
            mse_final=mse_final,
            counties=counties,
            column=diff_column,
            plot_arrows=False,
            cmap="RdBu",
            arrow_color="gray",
            title="MSE difference: " + model_a + " - " + model_b,
        )
    if plot_lisa:
        plot_county_lisa_map(
            transmit_matrix=transmit_matrix,
            mse_final=mse_final,
            counties=counties,
            column="mean_mse",
            permutations=999,
            sig=0.05,
            figsize=(10, 7),
            title="LISA MSE cluster map " + model_a + " - " + model_b,
            add_basemap=True,
            plot_arrows=False,
        )
    
    # Population slope plot and stats
    pop_stats = dict(beta=np.nan, p_beta=np.nan)
    if plot_populations:
        pop_stats = plot_centrality_vs_mse_seaborn_stats(
            mse_final,
            "population",
            mse_col=diff_column,
            kind="hex",
            log_transform="x",
            plot_figure=False,
        )

    # Build one row for the big summary table
    beta = pop_stats["beta"]
    p_beta = pop_stats["p_beta"]
    beta_stars = p_to_stars(p_beta)

    if np.isnan(beta):
        beta_display = ""
    else:
        beta_display = f"{beta:.4f}{beta_stars}"

    row = dict(
        ID=hypothesis_id,
        model_a=model_a,
        model_b=model_b,
        mse_comparison_plain=comparison_summary["comparison_plain"],
        mse_comparison_latex=comparison_summary["comparison_latex"],
        effect_rrb=comparison_summary["rank_biserial"],
        pop_beta=beta,
        pop_beta_p=p_beta,
        pop_beta_stars=beta_stars,
        pop_beta_display=beta_display,  # <- for pretty printing in pandas
    )
    return row


# Comparing different models

In [ ]:
ALL_COMPARISONS = [
    # A1 block — H1
    ("H1", "M1", "M4"),
    # ("H1", "M2", "M5"),
    # ("H1", "M3", "M6"),

    # # # A2 block — H2
    # ("H2", "M1", "M7"),
    # ("H2", "M2", "M8"),
    # ("H2", "M3", "M9"),

    # # # A3 block — H3
    # ("H3", "M4", "M7"),
    # ("H3", "M5", "M8"),
    # ("H3", "M6", "M9"),

    # # # Direct comparisons within time-series additions — H4
    # ("H4", "M1", "M2"),
    # ("H4", "M1", "M3"),
    # ("H4", "M2", "M3"),

    # # Direct comparisons within time-series additions — H4
    # ("H5", "M4", "M5"),
    # ("H5", "M5", "M6"),
    # ("H5", "M4", "M6"),

    # # Cross-architecture comparison — H5–H7
    # ("H6", "M1", "M10"),
    # ("H7", "M2", "M11"),
    # ("H8", "M3", "M11"),

    # # # Enhanced connectivity/leverage comparison — H8–H10
    # ("H9", "M4", "M12"),
    # ("H10", "M5", "M13"),
    # ("H11", "M6", "M13"),
]

BASE_KW = dict(
    base_path = r"region_mse", # the folder with all the region_mse files
    model_specifics = "_adamw_0.0001_rmse_zscore_RescalingOfAdjs_NewTsButterNoLog_16folds.shp",
    transmit_csv=r"../../3_Data/utils/dist.csv",
    fips_txt=r"../../3_Data/utils/county_fips_mainland.txt",
    counties_shp=r"../../3_Data/utils/county_data.shp",
    plot_maps=True,
    print_latex=True,
    print_dist_comp=True,
    plot_distributions=True,
    plot_histogram=True,
    plot_scatter=True,
    plot_populations=True,
    plot_lisa=True
)
        

all_rows = []
for hid, m1, m2 in ALL_COMPARISONS:
    print(f"\n===== Analyzing {m1} vs {m2} ({hid}) =====\n")
    row = analyze2_collect_stats(
        m1, m2,
        hypothesis_id=hid,
        **BASE_KW
    )
    all_rows.append(row)


In [ ]:
summary_df = pd.DataFrame(all_rows)
print(summary_df[["ID", "mse_comparison_plain", "effect_rrb", "pop_beta", "pop_beta_p"]])

# LaTeX table
latex_table = build_comparison_summary_latex(all_rows)
print("\nLaTeX summary table:\n")
print(latex_table)